# Simple Transformer LLM (code from this repo)

This notebook is a **tiny decoder Transformer**, the same idea as GPT, with only a few coding parts:

1. Load Python files from this project
2. Map characters to IDs
3. Train next-character prediction with **causal attention**
4. Generate text from a prompt like `def `

It is small on purpose so it can run on a laptop. Output will look like messy Python, not a real assistant.

In [1]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Embedding,
    Layer,
    LayerNormalization,
    MultiHeadAttention,
)

tf.random.set_seed(42)
np.random.seed(42)

SEQ_LEN = 32
EMBED_DIM = 32
NUM_HEADS = 4
FF_DIM = 64
N_LAYERS = 2
DROPOUT = 0.1
EPOCHS = 12
BATCH_SIZE = 64
MAX_CHARS = 40_000
STRIDE = 2

HERE = Path.cwd()
REPO_ROOT = HERE if HERE.name != "Tensorflow" else HERE.parent

## 1. Load Python from this repo

Starts with `simple_code_corpus.txt`, then appends `.py` files until about 40,000 characters.

In [2]:
def load_text(path):
    return path.read_text(encoding="utf-8", errors="ignore").replace("\r\n", "\n").lower()

chunks = []
seed = HERE / "simple_code_corpus.txt"
if not seed.exists():
    seed = REPO_ROOT / "Tensorflow" / "simple_code_corpus.txt"
chunks.append(load_text(seed))

for path in sorted(REPO_ROOT.rglob("*.py")):
    if path.name == "simple_llm.py":
        continue
    chunks.append(load_text(path))
    if sum(len(part) for part in chunks) >= MAX_CHARS:
        break

text = "\n\n".join(chunks)[:MAX_CHARS]
print("corpus chars:", len(text))
print(text[:500])

corpus chars: 40000
def add(a, b):
    return a + b

def greet(name):
    print("hello", name)

for i in range(5):
    print(i)

if x > 0:
    print("positive")
else:
    print("negative")

numbers = [1, 2, 3, 4]
total = 0
for n in numbers:
    total = total + n
print(total)

def square(n):
    return n * n

x = 10
y = 20
print(x + y)


"""patches that are applied at runtime to the virtual environment."""

from __future__ import annotations

import contextlib
import os
import sys

virtualenv_patch_file = os.path.ab


## 2. Character vocabulary

In [3]:
chars = sorted(set(text))
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for ch, i in char_to_id.items()}
print("vocab size:", len(chars))
print("characters:", repr("".join(chars)[:80]))

vocab size: 65
characters: '\n !"#%\'()*+,-./0123456789:<=>?@[\\]_`abcdefghijklmnopqrstuvwxyz{|}'


## 3. Next-character windows

`x` is 32 characters. `y` is the same window shifted by one. Every position learns the next character, like a small GPT.

In [4]:
ids = [char_to_id[ch] for ch in text]
x, y = [], []
for i in range(0, len(ids) - SEQ_LEN - 1, STRIDE):
    x.append(ids[i : i + SEQ_LEN])
    y.append(ids[i + 1 : i + SEQ_LEN + 1])

x = np.array(x, dtype=np.int32)
y = np.array(y, dtype=np.int32)
print("windows:", len(x), "x:", x.shape, "y:", y.shape)

windows: 19984 x: (19984, 32) y: (19984, 32)


## 4. Tiny Transformer

- token + position embeddings
- 2 blocks of causal `MultiHeadAttention` + feed-forward
- softmax over the vocabulary at every position

In [5]:
class AddPosition(Layer):
    def __init__(self, seq_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = Embedding(seq_len, embed_dim)

    def call(self, x):
        positions = tf.range(tf.shape(x)[1])
        return x + self.pos_emb(positions)


class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout, **kwargs):
        super().__init__(**kwargs)
        self.attn = MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(dropout)
        self.drop2 = Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.attn(x, x, use_causal_mask=True, training=training)
        x = self.norm1(x + self.drop1(attn_out, training=training))
        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out, training=training))


inputs = Input(shape=(SEQ_LEN,), dtype="int32")
h = Embedding(len(chars), EMBED_DIM)(inputs)
h = AddPosition(SEQ_LEN, EMBED_DIM)(h)
for _ in range(N_LAYERS):
    h = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)(h)
outputs = Dense(len(chars), activation="softmax")(h)

model = Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 32, 32)         │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_position (AddPosition)      │ (None, 32, 32)         │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 32, 32)         │         8,544 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 32, 32)         │         8,544 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32, 65)         │         2,145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,337 (87.25 KB)

 Trainable params: 22,337 (87.25 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train

In [6]:
history = model.fit(x, y, epochs=EPOCHS, batch_size=BATCH_SIZE)

Epoch 1/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - accuracy: 0.3386 - loss: 2.5138
Epoch 2/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.4093 - loss: 2.1272
Epoch 3/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.4666 - loss: 1.9363
Epoch 4/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.5065 - loss: 1.7977
Epoch 5/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.5349 - loss: 1.7012
Epoch 6/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.5540 - loss: 1.6319
Epoch 7/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.5651 - loss: 1.5848
Epoch 8/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.5768 - loss: 1.5427
Epoch 9/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.5853 - loss: 1.5107
Epoch 10/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.5911 - loss: 1.4855
Epoch 11/12
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.5968 - loss: 1.4641
Epoch 12/12
313/313 ━━━━━━━━━━━

## 6. Generate from a prompt

The model scores the **last** position in the window, samples a character, then slides forward.

In [10]:
def generate(prompt, n_chars=160):
    seed = prompt.lower()
    missing = [ch for ch in seed if ch not in char_to_id]
    if missing:
        raise ValueError(f"unseen characters in prompt: {missing}")
    if len(seed) < SEQ_LEN:
        seed = seed.rjust(SEQ_LEN)
    else:
        seed = seed[-SEQ_LEN:]
    generated = seed
    for _ in range(n_chars):
        window = generated[-SEQ_LEN:]
        inp = np.array([[char_to_id[ch] for ch in window]], dtype=np.int32)
        probs = model.predict(inp, verbose=0)[0, -1]
        next_id = int(np.random.choice(len(probs), p=probs))
        generated += id_to_char[next_id]
    return generated.lstrip()

print(generate("loop"))

loopath is atobject",
    "toorlure",
    "conunitionstyypescalditericatetionly,
           hmetanss
.   tren = :
    """
             ""
    "scaleinit",
    "offi


Same model as a script:

```text
python Tensorflow/simple_llm.py
```